# Stage 1: Select Implied Copyable Trades

Find profitable follower BUYs that follow a leader's BUY or SELL on the same
token within a time window. Two separate leader groups: **buy leaders** (whose
BUYs precede follower BUYs) and **sell leaders** (whose SELLs precede follower
BUYs).

Grid-search over selection thresholds to maximize **copyable PnL from implied
trades** on the validation split.

**Output:** `stage1_implied_result.json` with best selection params.

In [125]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    select_follower_wallets,
    select_leader_wallets,
    detect_implied_buys,
    score_leaders,
    evaluate_implied_pnl,
    run_implied_grid_search,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [126]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full)

Markets: 1877548
Filtered markets for {'Weather'}: 87967
Loading 16 trade shards...
Total trades loaded: 13,603,198
Unique wallets: 4,054
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-22 05:47:10+00:00
Train:     6,619,486 trades  (24,504 markets)
Val:       4,081,660 trades  (16,109 markets)
Test:      2,902,052 trades  (13,119 markets)
Total:    13,603,198 trades  (53,732 markets)


## Compute wallet metrics on training data

In [127]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3514


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.0236,NaN,13.2971,0.0193,45
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0046,0.0020,-2.5788,0.0000,304
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0193,0.0021,84.8491,-0.0031,9771
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,NaN,0.0000,NaN,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.6776,-1.0000,115.4494,0.4865,34
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,NaN,-20.9264,NaN,299
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0128,-1.0000,28.7030,-0.1038,63
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0100,-0.2460,38.8680,-0.0018,5701
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.1991,-0.0749,-64.3190,0.0000,1473
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.0036,-0.9749,-87.4111,0.0000,3279


## Baseline selection

In [128]:
follower_wallets = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=0.05,
    min_trade_value=100,
    min_open_buys=10,
)
print(f"Followers: {len(follower_wallets)}")

buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=20,
    min_roi=None,
    max_market_pnl_hhi=1,
    side="BUY",
)
print(f"Buy leaders: {len(buy_leaders)}")

sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=20,
    min_roi=None,
    max_market_pnl_hhi=1,
    side="SELL",
)
print(f"Sell leaders: {len(sell_leaders)}")

Followers: 235
Buy leaders: 1382
Sell leaders: 1382


## Baseline evaluation

In [133]:
follower_ws = set(follower_wallets['wallet'])
buy_leader_ws = set(buy_leaders['wallet'])
sell_leader_ws = set(sell_leaders['wallet'])

for split_name, df_split in  [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(
        df_split, follower_ws, buy_leader_ws,
        time_window_minutes=5, leader_side="BUY"
    )
    sell_ev = evaluate_implied_pnl(
        df_split, follower_ws, sell_leader_ws,
        time_window_minutes=5, leader_side="SELL",
    )
    total = buy_ev["copyable_pnl"] + sell_ev["copyable_pnl"]
    print(f"{split_name}: buy_pnl={buy_ev['copyable_pnl']:.2f} ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)  "
          f"sell_pnl={sell_ev['copyable_pnl']:.2f} ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)  "
          f"total={total:.2f}")

TRAIN: buy_pnl=92525.00 (58754 trades, 1090 leaders)  sell_pnl=91252.23 (61090 trades, 819 leaders)  total=183777.23
VAL: buy_pnl=14571.95 (30710 trades, 667 leaders)  sell_pnl=15085.65 (29837 trades, 523 leaders)  total=29657.61
TEST: buy_pnl=15431.53 (18312 trades, 514 leaders)  sell_pnl=22523.58 (18327 trades, 376 leaders)  total=37955.12


## Score leaders (baseline)

In [135]:
# Buy leaders
buy_implied = detect_implied_buys(
    df_train, follower_ws, buy_leader_ws,
    time_window_minutes=5, leader_side="BUY",
)
buy_scores = score_leaders(buy_implied)
print("Top buy leaders:")
print(buy_scores.head(10).to_string())

print()

# Sell leaders
sell_implied = detect_implied_buys(
    df_train, follower_ws, sell_leader_ws,
    time_window_minutes=5, leader_side="SELL",
)
sell_scores = score_leaders(sell_implied)
print("Top sell leaders:")
print(sell_scores.head(10).to_string())

Top buy leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0xb06a0eae498750ed0acac7e1f759f741c56e52f5            152                    8371.0513                 2287           1305
1  0x3dc44175ae2d0175ee7bc76cd9ec04b614a91fd8             40                    4015.0235                   89             67
2  0x9cb35555b913c805a62a2e34bce4ef14f0e81367            139                    3516.5130                 1292            704
3  0x26123cbf0f4820f7e70408a8c054ba7615c05289            158                    2693.9643                 2685           1245
4  0x48180cfe026031c280ebdea2acad996867cde5de             70                    2330.7459                  308            186
5  0x945a49252f772a10c6ddd1d1e1e24ee20438a48c            164                    2266.9639                 2557           1202
6  0xde0fa43faad1c0da4881c63681ad07a8c850a4e7              5                    2199.6912            

## Grid search

Vary selection thresholds to maximize copyable PnL from implied trades on the
validation split.

In [136]:
param_grid = dict(
    # Follower selection
    min_follower_copyable_roi=[0, 0.05, 0.10],
    min_follower_trade_value=[10, 100],
    min_follower_open_buys=[10],
    # Buy leader selection
    min_buy_leader_trade_count=[20],
    min_buy_leader_roi=[0.0],
    max_buy_leader_hhi=[1],
    # Sell leader selection
    min_sell_leader_trade_count=[20],
    min_sell_leader_roi=[-0.1],
    max_sell_leader_hhi=[1],
    # Detection
    time_window_minutes=[10],
    min_pair_interactions=[0],
)

n_combos = np.prod([len(v) for v in param_grid.values()])
print(f"Grid: {n_combos:.0f} combos")

Grid: 6 combos


In [141]:
res_df = run_implied_grid_search(param_grid, wallet_vol, df_val)
res_df.head()

Grid: 6 combos, 8 workers
  [6/6] 12.1s elapsed
Done: 6 configs in 12.1s


,min_follower_copyable_roi,min_follower_trade_value,min_follower_open_buys,min_buy_leader_trade_count,min_buy_leader_roi,max_buy_leader_hhi,min_sell_leader_trade_count,min_sell_leader_roi,max_sell_leader_hhi,time_window_minutes,min_pair_interactions,error,implied_copyable_pnl
0,0.1000,10,10,20,0.0000,1,20,-0.1000,1,10,0,'total_copyable_pnl',-inf
1,0.1000,100,10,20,0.0000,1,20,-0.1000,1,10,0,'total_copyable_pnl',-inf
2,0.0500,10,10,20,0.0000,1,20,-0.1000,1,10,0,'total_copyable_pnl',-inf
3,0.0500,100,10,20,0.0000,1,20,-0.1000,1,10,0,'total_copyable_pnl',-inf
4,0.0000,100,10,20,0.0000,1,20,-0.1000,1,10,0,'total_copyable_pnl',-inf


## Grid search results

In [142]:
best_row = res_df.iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}

print(f"Best config (val implied_pnl={best_row['implied_copyable_pnl']:.2f}):")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"  followers={best_row['followers']:.0f}  buy_leaders={best_row['buy_leaders']:.0f}  sell_leaders={best_row['sell_leaders']:.0f}")
print(f"  buy_pnl={best_row['buy_pnl']:.2f}  sell_pnl={best_row['sell_pnl']:.2f}")

Best config (val implied_pnl=-inf):
  min_follower_copyable_roi: 0.1
  min_follower_trade_value: 10
  min_follower_open_buys: 10
  min_buy_leader_trade_count: 20
  min_buy_leader_roi: 0.0
  max_buy_leader_hhi: 1
  min_sell_leader_trade_count: 20
  min_sell_leader_roi: -0.1
  max_sell_leader_hhi: 1
  time_window_minutes: 10
  min_pair_interactions: 0


KeyError: 'followers'

In [ ]:
print("Top 10 results:")
res_df[["implied_copyable_pnl", "buy_pnl", "sell_pnl", "buy_trades", "sell_trades",
        "followers", "buy_leaders", "sell_leaders", "time_window_minutes"]].head(10)

Top 10 results:


,implied_copyable_pnl,buy_pnl,sell_pnl,buy_trades,sell_trades,followers,buy_leaders,sell_leaders,time_window_minutes
1,30665.3406,14725.4872,15939.8534,37511,37518,235,651,436,10
0,23424.6533,10740.1139,12684.5395,24160,24151,167,598,408,10
2,20156.4201,8961.4188,11195.0013,144597,118088,496,744,505,10


## Evaluate best config on all splits

In [144]:
# Re-select wallets with best params
best_followers = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=best_params["min_follower_copyable_roi"],
    min_trade_value=best_params["min_follower_trade_value"],
    min_open_buys=best_params["min_follower_open_buys"],
)
best_buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=best_params["min_buy_leader_trade_count"],
    min_roi=best_params["min_buy_leader_roi"],
    max_market_pnl_hhi=best_params["max_buy_leader_hhi"],
    side="BUY",
)
best_sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=best_params["min_sell_leader_trade_count"],
    min_roi=best_params["min_sell_leader_roi"],
    max_market_pnl_hhi=best_params["max_sell_leader_hhi"],
    side="SELL",
)

b_fw = set(best_followers["wallet"])
b_blw = set(best_buy_leaders["wallet"])
b_slw = set(best_sell_leaders["wallet"])
tw = best_params["time_window_minutes"]
min_pi = best_params["min_pair_interactions"]

print(f"Followers: {len(b_fw)}  Buy leaders: {len(b_blw)}  Sell leaders: {len(b_slw)}")
print()

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(df_split, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
    sell_ev = evaluate_implied_pnl(df_split, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")

    buy_pnl = buy_ev["copyable_pnl"] if buy_ev["trade_count"] >= min_pi else 0.0
    sell_pnl = sell_ev["copyable_pnl"] if sell_ev["trade_count"] >= min_pi else 0.0
    total = buy_pnl + sell_pnl

    print(f"{split_name}: buy_pnl={buy_pnl:.2f} ({buy_ev['trade_count']} trades)  "
          f"sell_pnl={sell_pnl:.2f} ({sell_ev['trade_count']} trades)  "
          f"total={total:.2f}")

Followers: 176  Buy leaders: 1263  Sell leaders: 841

TRAIN: buy_pnl=76336.56 (43457 trades)  sell_pnl=77045.74 (46501 trades)  total=153382.30
VAL: buy_pnl=10800.78 (25398 trades)  sell_pnl=12631.23 (25199 trades)  total=23432.01
TEST: buy_pnl=10438.77 (15536 trades)  sell_pnl=14917.14 (15395 trades)  total=25355.91


## Save stage 1 result

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# Collect wallet records for each group
wallet_cols = [
    "wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "market_pnl_hhi",
]

def _wallet_records(df):
    if df is None or df.empty:
        return []
    cols = [c for c in wallet_cols if c in df.columns]
    records = df[cols].to_dict(orient="records")
    return [{k: _convert(v) for k, v in w.items()} for w in records]


metadata = {
    "type": "implied",
    "tags": ["Weather"],
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_followers": len(b_fw),
    "n_buy_leaders": len(b_blw),
    "n_sell_leaders": len(b_slw),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "type": "implied",
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_implied_copyable_pnl": float(best_row["implied_copyable_pnl"]),
    "metadata": metadata,
    "wallets": {
        "followers": _wallet_records(best_followers),
        "buy_leaders": _wallet_records(best_buy_leaders),
        "sell_leaders": _wallet_records(best_sell_leaders),
    },
}

out_path = Path("stage1_implied_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path}")

Saved stage 1 implied result -> stage1_implied_result.json
